In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition, ToolNode
from pydantic import BaseModel
from langgraph.types import Command, interrupt
from IPython.display import display, Image
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal
from pprint import pprint

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

/Users/amitrana/Documents/work/ai/AI-Creativity-Hub/backend/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# States
class Email(BaseModel):
    subject: str
    body: str


class EmailState(BaseModel):
    topic: str
    draft: Email = Email(subject="", body="")
    feedback: str = ""
    final_summary: str = ""
    to: str = ""


# Prompt
email_prompt = PromptTemplate(
    template="""You're email assistant which help to draft emails, 
                please keep email concise and 4-5 sentances only. 
                User will ask Topic: {topic}.
                You'll draft professional email subject and body and return a structured JSON format.
                If Draft: {draft} have some values then it means user is asking to refine. So that you need to pick existing and refine.
            """
)

: 

In [ ]:
def create_draft(state: EmailState):
    structured_llm = llm.with_structured_output(Email)
    prompt_text = email_prompt.format(
        topic=state.topic, draft=state.draft
    )
    result = structured_llm.invoke([HumanMessage(content=prompt_text)])
    return {"draft": result}

def human_feedback(state: EmailState):
    fb = interrupt("Do you want refinement yes or no?")
    return {"feedback": fb}

def handle_condition(state: EmailState)-> Literal["create_draft", "ask_for_recipient"]:
    return  "create_draft" if "yes" in state.feedback else "ask_for_recipient" 


def ask_for_recipient(state: EmailState):
    return {}

def send_email(state: EmailState):
    """Tool to send email"""
    return {"final_summary": "Email is sent to User"}
    
memory = MemorySaver()
builder = StateGraph(EmailState)
builder.add_node("create_draft", create_draft)
builder.add_node("human_feedback", human_feedback)
builder.add_node("ask_for_recipient", ask_for_recipient)
builder.add_node("send_email", send_email)

builder.add_edge(START, "create_draft")
builder.add_edge("create_draft", "human_feedback")
builder.add_conditional_edges("human_feedback", handle_condition)
builder.add_edge("ask_for_recipient", "send_email")
builder.add_edge("send_email", END)

graph = builder.compile(checkpointer=memory, interrupt_before=["ask_for_recipient"])

display(Image(graph.get_graph().draw_mermaid_png()))

thread = {"configurable": {"thread_id": "draft_1"}}

# First invoke
print(f"\n\n{'=='*10} First Invoke {'=='*10}\n\n")
result = graph.invoke({"topic": "Write email to ask for appraisal"}, config=thread)
print(result)

# Second invoke: Refinement
print(f"\n\n{'=='*10} Second Invoke: Need Refinement {'=='*10}\n\n")
result = graph.invoke(Command(resume="yes"), config=thread)
print(result)

# Invoke: Approved
print(f"\n\n{'=='*10} Invoke: Approved {'=='*10}\n\n")
result = graph.invoke(Command(resume="no"), config=thread)
print(result)

# Invoke: Ask for recepient and send email
print(f"\n\n{'=='*10} Final Invoke: Ask for recepient and send email {'=='*10}\n\n")
email = "someone@example.com"
graph.update_state(thread, {"to": email}, as_node="ask_for_recipient")
result = graph.invoke(None, config=thread)
pprint(result)




: 